# Project 23 — Retrieval Reranking Lab

## Compare No-Rerank vs LLM-Based Reranking

**Goal:** Demonstrate that a two-stage pipeline — fast initial retrieval
followed by precision reranking — produces better answers than retrieval
alone. We compare raw vector similarity, LLM-based relevance scoring,
and an LLM listwise reranker, then measure the impact on downstream QA.

**Stack:** Ollama · LangChain · ChromaDB · Jupyter

```
Query ──► Dense Retriever (top-k=5) ──► Reranker (LLM) ──► top-3 ──► LLM ──► Answer
```

### What You'll Learn

1. Why initial retrieval order is often sub-optimal
2. LLM-based pointwise relevance scoring
3. Listwise reranking with a single LLM call
4. Side-by-side comparison of ranking quality
5. Latency vs quality tradeoff analysis

### Prerequisites

- **Ollama** running locally with `nomic-embed-text` and `qwen3:8b`
- Python 3.9+

In [ ]:
!pip install -q langchain langchain-ollama langchain-community chromadb

## Step 1 — Verify Ollama Is Running

In [ ]:
import requests
OLLAMA_BASE = "http://localhost:11434"
try:
    r = requests.get(f"{OLLAMA_BASE}/api/tags", timeout=5)
    r.raise_for_status()
    print(f"✅ Ollama is running — {len(r.json().get('models',[]))} model(s)")
except Exception as e:
    print(f"❌ Cannot reach Ollama: {e}")

## Step 2 — Build Corpus and Retriever

A Python-focused corpus where several documents are semantically close,
making reranking valuable for distinguishing the *most* relevant one.

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.schema import Document
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import time

llm = ChatOllama(model="qwen3:8b", temperature=0.0)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

corpus = [
    Document(page_content="Python list comprehensions provide a concise way to create lists. "
        "The syntax is [expr for item in iterable if condition].", metadata={"id": 1, "topic": "list_comp"}),
    Document(page_content="Python dictionaries store key-value pairs. Common methods: "
        "get(), keys(), values(), items(), update().", metadata={"id": 2, "topic": "dicts"}),
    Document(page_content="Python generators use yield to produce values lazily, saving "
        "memory for large datasets. Generator expressions use parentheses.", metadata={"id": 3, "topic": "generators"}),
    Document(page_content="Python decorators are functions that modify other functions. "
        "Common use cases: logging, caching, authentication, timing.", metadata={"id": 4, "topic": "decorators"}),
    Document(page_content="Python context managers handle setup and cleanup with the with "
        "statement. Implement via __enter__/__exit__ or @contextmanager.", metadata={"id": 5, "topic": "context_mgr"}),
    Document(page_content="Python async/await enables concurrent I/O operations. asyncio "
        "provides event loop, tasks, and coroutine management.", metadata={"id": 6, "topic": "async"}),
    Document(page_content="Python type hints improve code documentation: def greet(name: str) "
        "-> str. Use mypy for static type checking.", metadata={"id": 7, "topic": "types"}),
    Document(page_content="Python dataclasses reduce boilerplate for data containers. "
        "Use @dataclass decorator. Supports defaults, ordering, freezing.", metadata={"id": 8, "topic": "dataclass"}),
    Document(page_content="Python itertools provides memory-efficient tools for iteration: "
        "chain, product, combinations, permutations, groupby.", metadata={"id": 9, "topic": "itertools"}),
    Document(page_content="Python functools includes higher-order functions: lru_cache for "
        "memoization, partial for argument binding, reduce for accumulation.", metadata={"id": 10, "topic": "functools"}),
]

vectorstore = Chroma.from_documents(corpus, embeddings, collection_name="rerank_lab")
print(f"✅ Corpus indexed: {len(corpus)} documents")

## Step 3 — Initial Retrieval (No Reranking)

Raw vector similarity retrieval returns top-5 candidates. These are
often *roughly* right but not perfectly ordered by relevance.

In [ ]:
def retrieve_no_rerank(query, k=5):
    return vectorstore.similarity_search_with_score(query, k=k)

query = "How do I efficiently process large amounts of data in Python?"

print(f"Query: {query}\n")
print("=== NO RERANKING (raw vector similarity) ===")
no_rerank_results = retrieve_no_rerank(query)
for i, (doc, score) in enumerate(no_rerank_results):
    print(f"  [{i+1}] score={score:.4f} id={doc.metadata['id']} "
          f"topic={doc.metadata['topic']} — {doc.page_content[:55]}...")

## Step 4 — Pointwise LLM Reranker

For each retrieved document, we ask the LLM: *"How relevant is this
document to the query?"* on a 0-10 scale. This is slower but much
more accurate than embedding similarity alone.

In [ ]:
rerank_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a relevance judge. Given a query and a document,
rate the document's relevance on a scale of 0-10.
Return ONLY the number, nothing else.

0-2: Not relevant at all
3-5: Somewhat relevant
6-8: Relevant
9-10: Highly relevant and directly answers the query"""),
    ("human", "Query: {query}\n\nDocument: {document}\n\nRelevance score (0-10):")
])
rerank_chain = rerank_prompt | llm | StrOutputParser()

def rerank_pointwise(query, initial_results, top_k=3):
    """Score each document individually and re-sort."""
    scored = []
    for doc, vec_score in initial_results:
        try:
            relevance = rerank_chain.invoke({"query": query, "document": doc.page_content})
            score = float(relevance.strip().split()[0])
        except (ValueError, IndexError):
            score = 0.0
        scored.append((doc, score, vec_score))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]

print("=== POINTWISE LLM RERANKING ===")
t0 = time.time()
reranked = rerank_pointwise(query, no_rerank_results)
rerank_time = time.time() - t0
for i, (doc, llm_score, vec_score) in enumerate(reranked):
    print(f"  [{i+1}] llm={llm_score:.0f} vec={vec_score:.4f} "
          f"topic={doc.metadata['topic']} — {doc.page_content[:55]}...")
print(f"  ⏱ Reranking took {rerank_time:.1f}s")

## Step 5 — Listwise LLM Reranker

Instead of scoring each document individually (N LLM calls), we can
present all candidates at once and ask the LLM to rank them in a single
call. This is faster but may be less precise.

In [ ]:
listwise_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a search relevance expert. Given a query and a list of documents, "
     "rank them from most to least relevant. Return ONLY the document numbers in order, "
     "separated by commas. Example: 3,1,5,2,4"),
    ("human", "Query: {query}\n\nDocuments:\n{documents}\n\nRanking (most to least relevant):")
])
listwise_chain = listwise_prompt | llm | StrOutputParser()

def rerank_listwise(query, initial_results, top_k=3):
    """Rank all documents in a single LLM call."""
    docs_text = "\n".join([
        f"Document {i+1}: {doc.page_content}" for i, (doc, _) in enumerate(initial_results)
    ])
    try:
        ranking = listwise_chain.invoke({"query": query, "documents": docs_text})
        order = [int(x.strip()) - 1 for x in ranking.strip().split(",") if x.strip().isdigit()]
        reranked = [(initial_results[i][0], len(initial_results) - rank, initial_results[i][1])
                    for rank, i in enumerate(order) if i < len(initial_results)]
    except Exception:
        reranked = [(doc, 0, score) for doc, score in initial_results]
    return reranked[:top_k]

print("=== LISTWISE LLM RERANKING ===")
t0 = time.time()
listwise_results = rerank_listwise(query, no_rerank_results)
listwise_time = time.time() - t0
for i, (doc, rank_score, vec_score) in enumerate(listwise_results):
    print(f"  [{i+1}] topic={doc.metadata['topic']} — {doc.page_content[:55]}...")
print(f"  ⏱ Listwise reranking took {listwise_time:.1f}s")

## Step 6 — Side-by-Side Comparison Across Queries

In [ ]:
test_queries = [
    "How do I efficiently process large amounts of data in Python?",
    "What's the best way to add caching to my functions?",
    "How do I handle concurrent network requests?",
    "What's a clean way to define data structures?",
    "How do I iterate over combinations of items?",
]

print(f"{'Query':<55} {'Raw':>5} {'Point':>6} {'List':>5}")
print("-" * 75)

for q in test_queries:
    raw = retrieve_no_rerank(q, k=5)
    raw_top = raw[0][0].metadata["topic"]

    pt = rerank_pointwise(q, raw, top_k=1)
    pt_top = pt[0][0].metadata["topic"]

    lt = rerank_listwise(q, raw, top_k=1)
    lt_top = lt[0][0].metadata["topic"]

    changed_p = "↑" if pt_top != raw_top else "="
    changed_l = "↑" if lt_top != raw_top else "="
    print(f"{q[:53]:<55} {raw_top[:5]:>5} {pt_top[:5]:>5}{changed_p} {lt_top[:5]:>4}{changed_l}")

## Step 7 — End-to-End QA: No-Rerank vs Rerank

In [ ]:
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using ONLY the provided context. Be concise."),
    ("human", "Context: {context}\n\nQuestion: {question}")
])
qa_chain = qa_prompt | llm | StrOutputParser()

q = "How do I process large datasets efficiently in Python?"
print(f"Q: {q}\n")

# Without reranking
raw = vectorstore.similarity_search(q, k=3)
ctx_raw = "\n".join([d.page_content for d in raw])
ans_raw = qa_chain.invoke({"context": ctx_raw, "question": q})
print(f"[NO RERANK] Sources: {[d.metadata['topic'] for d in raw]}")
print(f"  {str(ans_raw)[:200]}\n")

# With reranking
raw5 = retrieve_no_rerank(q, k=5)
reranked = rerank_pointwise(q, raw5, top_k=3)
ctx_re = "\n".join([d.page_content for d, _, _ in reranked])
ans_re = qa_chain.invoke({"context": ctx_re, "question": q})
print(f"[RERANKED]  Sources: {[d.metadata['topic'] for d, _, _ in reranked]}")
print(f"  {str(ans_re)[:200]}")

## Limitations and Tradeoffs

1. **Latency cost:** Pointwise reranking makes N LLM calls per query.
   Listwise uses 1 call but may be less precise for large N.
2. **LLM judge reliability:** The reranker LLM may have biases
   (preferring longer documents, first-position bias, etc.).
3. **No cross-encoder:** A proper cross-encoder reranker (e.g., from
   sentence-transformers) would be faster and more accurate.
4. **Small corpus:** With 10 documents, reranking impact is modest.
   Benefits are much larger with hundreds of candidates.

## How to Extend This Project

1. **Cross-encoder reranker:** Use `sentence-transformers` CrossEncoder
   for faster, learned reranking.
2. **Cascade pipeline:** BM25 (top-100) → dense (top-20) → reranker (top-5).
3. **Reranker evaluation:** Use nDCG with human relevance labels.
4. **Latency budget:** Set a latency ceiling and pick the best method
   that fits within it.

## What You Learned

| Concept | What We Did |
|---|---|
| **Vector similarity baseline** | Fast but imprecise initial ranking |
| **Pointwise LLM reranking** | Score each document individually |
| **Listwise LLM reranking** | Rank all documents in one call |
| **Side-by-side comparison** | Which queries benefit from reranking |
| **Latency measurement** | Cost of reranking in seconds |
| **End-to-end QA impact** | Better ranking → better answers |